<a href="https://colab.research.google.com/github/Killian091/Machine_Learning/blob/main/Backpropogation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **BCKPROPOGATION -> Using formulaes**

This notebook demonstrates a simple implementation of a neural network (or a linear model in this case, as there's no activation function between layers) using NumPy for a regression task. It covers parameter initialization, a forward pass, and a basic parameter update rule, resembling a form of backpropagation without explicit gradient calculation in a generalized way, but rather specific updates for a 2-layer network.

In [ ]:
import pandas as pd
import numpy as np

This cell imports the necessary libraries:
- `pandas` is used for data manipulation, particularly for creating and handling DataFrames.
- `numpy` is used for numerical operations, especially for array manipulations and mathematical calculations, which are fundamental for neural network implementations.

In [ ]:
df = pd.DataFrame([[8,8,4],[7,9,5],[6,10,6],[5,12,7]], columns=['cgpa','profile_score','lpa'])

Here, a pandas DataFrame `df` is created. This DataFrame contains sample data with three columns: `cgpa`, `profile_score`, and `lpa`. It seems like `cgpa` and `profile_score` are input features, and `lpa` (Lakh Per Annum, likely a salary or package) is the target variable to be predicted.

In [ ]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


This cell simply displays the created DataFrame `df`, showing its content and structure.

In [ ]:
def initialize_parameters(layers_dim):

  np.random.seed(3)
  parameters = {}
  L = len(layers_dim)

  for l in range(1,L):

    parameters['W' + str(l)] = np.ones((layers_dim[l-1], layers_dim[l]))*0.1
    parameters['b' + str(l)] = np.zeros((layers_dim[l],1))

  return parameters

The `initialize_parameters` function is responsible for setting up the initial weights (W) and biases (b) for a neural network.
- It takes `layers_dim` as input, which is a list specifying the number of neurons in each layer (e.g., `[input_dim, hidden_dim, output_dim]`).
- `np.random.seed(3)` ensures reproducibility of the random initialization.
- Weights `W` are initialized to small values (0.1) using `np.ones`, and biases `b` are initialized to zeros. The shape of `W` is `(neurons_in_prev_layer, neurons_in_current_layer)` and `b` is `(neurons_in_current_layer, 1)`.

In [ ]:
initialize_parameters([2,2,1])

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

This cell demonstrates the usage of the `initialize_parameters` function with `[2,2,1]`, indicating an input layer of 2 features, a hidden layer of 2 neurons, and an output layer of 1 neuron. It then prints the initialized weights and biases.

In [ ]:
def linear_forward(A_prev, W, b):
  Z = np.dot(W.T, A_prev) + b
  return Z

The `linear_forward` function performs the linear part of a neural network layer's forward propagation.
- It calculates `Z = W.T @ A_prev + b`, where `A_prev` is the activation from the previous layer, `W` is the weight matrix for the current layer, and `b` is the bias vector. Note that the weights `W` are transposed (`W.T`) to align with the matrix multiplication convention where input features are columns.

In [ ]:
# Forward Loop
def L_layer_forward(X, parameters):

  A = X
  L = len(parameters)//2

  for l in range (1, L+1):
    A_prev = A
    Wl = parameters['W' + str(l)]
    bl = parameters['b' + str(l)]

    A = linear_forward(A_prev, Wl, bl)

  return A,A_prev

The `L_layer_forward` function implements the full forward pass for a multi-layer network (in this case, up to L layers).
- It takes the input `X` and the `parameters` (weights and biases) as input.
- It iterates through each layer, applying the `linear_forward` function. Since there's no activation function mentioned after `linear_forward` in this code, it effectively performs a series of linear transformations.
- It returns the final output `A` (which is `y_hat` in the context of the problem) and `A_prev`, which is the output of the second to last layer (hidden layer in this 2-layer example) and is used in the `update_parameters` function for backpropagation-like updates.

In [ ]:
X = df[['cgpa', 'profile_score']].values[0].reshape(2,1)
y = df[['lpa']].values[0]

parameters = initialize_parameters([2,2,1])

y_hat, A1 = L_layer_forward(X, parameters)
y_hat = y_hat[0][0]

This cell prepares the input `X` and target `y` for a single training example.
- `X` is extracted from the first row of the DataFrame's 'cgpa' and 'profile_score' columns and reshaped into a `(2,1)` column vector.
- `y` is extracted from the first row of the 'lpa' column.
- It then initializes parameters for a `[2,2,1]` network and performs a forward pass using `L_layer_forward` to get `y_hat` (the prediction) and `A1` (the output of the hidden layer, before the final output layer).

In [ ]:
def update_parameters(parameters,y,y_hat,A1,X):
  parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001 * 2 * (y - y_hat)*A1[0][0])
  parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat)*A1[1][0])
  parameters['b2'][0][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat))

  parameters['W1'][0][0] = parameters['W1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0]*X[0][0])
  parameters['W1'][0][1] = parameters['W1'][0][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0]*X[1][0])
  parameters['b1'][0][0] = parameters['b1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0])

  parameters['W1'][1][0] = parameters['W1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0]*X[0][0])
  parameters['W1'][1][1] = parameters['W1'][1][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0]*X[1][0])
  parameters['b1'][1][0] = parameters['b1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0])

  return parameters

The `update_parameters` function performs a manual update of the network's weights and biases. This update rule is a simplified version of gradient descent for a specific 2-layer linear network, attempting to minimize the squared error `(y - y_hat)^2`.
- It directly modifies `W1`, `b1`, `W2`, and `b2` based on the error `(y - y_hat)`, the learning rate (0.001), and the intermediate activations/inputs (`A1`, `X`).
- The calculations shown are specific to this two-layer architecture and are derived from the chain rule for backpropagation, though not implemented in a general, vectorized way.

In [ ]:
parameters = update_parameters(parameters,y,y_hat,A1,X)

parameters

/tmp/ipykernel_2009/3292834232.py:2: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001 * 2 * (y - y_hat)*A1[0][0])
/tmp/ipykernel_2009/3292834232.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat)*A1[1][0])
/tmp/ipykernel_2009/3292834232.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  parameters['b2'][0][0] = parameters['W2'][1][0] + 

{'W1': array([[0.10658137, 0.10658137],
        [0.10658137, 0.10658137]]),
 'b1': array([[0.00082267],
        [0.00082267]]),
 'W2': array([[0.111776],
        [0.111776]]),
 'b2': array([[0.119136]])}

This cell calls the `update_parameters` function with the current parameters, target `y`, prediction `y_hat`, hidden layer output `A1`, and input `X` to update the network's parameters. It then prints the updated parameters.
- The `DeprecationWarning` messages indicate that you are trying to convert a NumPy array with more than one dimension to a scalar implicitly. This often happens when indexing an array that might sometimes contain a single element but is still treated as an array. It's good practice to ensure you're extracting a single element explicitly, e.g., by using `[0][0]` if you intend to get a scalar from a 2D array of shape (1,1).

In [ ]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 5

for i in range(epochs):

  Loss = []

  for j in range(df.shape[0]):

    X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
    y = df[['lpa']].values[j][0]

    # Parameter initialization


    y_hat,A1 = L_layer_forward(X,parameters)
    y_hat = y_hat[0][0]

    update_parameters(parameters,y,y_hat,A1,X)

    Loss.append((y-y_hat)**2)

  print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

parameters

Epoch -  1 Loss -  25.321744156025517
Epoch -  2 Loss -  18.320004165722047
Epoch -  3 Loss -  9.473661050729628
Epoch -  4 Loss -  3.2520938634031613
Epoch -  5 Loss -  1.3407132589299962


{'W1': array([[0.26507636, 0.38558861],
        [0.27800387, 0.40980287]]),
 'b1': array([[0.02749056],
        [0.02974394]]),
 'W2': array([[0.41165744],
        [0.48302736]]),
 'b2': array([[0.48646246]])}

This block implements the training loop for the neural network over a specified number of `epochs`.
- **Initialization**: Parameters are re-initialized at the beginning of the training.
- **Epoch Loop**: It iterates `epochs` times.
- **Data Loop**: Inside each epoch, it iterates through each training example in the DataFrame.
  - For each example, `X` and `y` are extracted.
  - A forward pass is performed to get `y_hat` and `A1`.
  - The `update_parameters` function is called to adjust the weights and biases based on the current example's error.
  - The squared error `(y - y_hat)**2` is calculated and added to a `Loss` list.
- **Loss Reporting**: After each epoch, the mean of the calculated losses for that epoch is printed.
- **Final Parameters**: Finally, the updated parameters after all epochs are printed. This shows how the network's weights and biases have adjusted over time to better fit the training data.